In [ ]:
import pandas as pd
import glob

# načtení a spojení číselníků krajů s vazbami na obce
kraj_72 = pd.read_csv("kraj_72_vazba.csv")
kraj_65 = pd.read_csv("kraj_65_vazba.csv")
kraje_vazba = pd.concat([kraj_72, kraj_65], ignore_index=True)
kraje_vazba = kraje_vazba[["text1", "chodnota2", "text2"]]

# načtení a spojení csv s počty ekonomických subjektů za jednotlivá časová období
odvetvi_roky = glob.glob("C:/Dominika/industrial_analysis/data/subjekty_2021-2025/subjekty/*.csv")
subjekty = pd.concat([pd.read_csv(f) for f in odvetvi_roky], ignore_index=True)

# merge subjektů a krajů, čištění území a kontrola
subjekty = subjekty.merge(
    kraje_vazba, left_on="vuzemi_kod", right_on="chodnota2", how="left"
)
subjekty = subjekty[~subjekty["vuzemi_cis"].isin([43, 44])]
subjekty = subjekty[subjekty["odvetvi_txt"].notna()]


assert subjekty["text1"].notna().all()

# odstranení nepotřebných sloupců
subjekty = subjekty[["casref", "odvetvi_txt", "text1", "hodnota"]]

# odstranění mezer
subjekty["odvetvi_txt"] = subjekty["odvetvi_txt"].str.strip()

# odstranění nadbytečných řádků
subjekty = subjekty[
    ~subjekty["odvetvi_txt"].isin(
        [
            "Nedefinováno",
            "Činnosti domácností jako zaměstnavatelů; činnosti domácností produkujících blíže neurčené výrobky a služby pro vlastní potřebu",
            "Činnosti exteritoriálních organizací a orgánů",
        ]
    )
]

# filtrování časových údajů z čtvrtletních na roční a přejmenování
subjekty = subjekty[subjekty["casref"].str.contains("12-31", na=False)]
subjekty["casref"] = subjekty["casref"].str[:4].astype(int)

# převedení hodnot na číselný formát
subjekty["hodnota"] = pd.to_numeric(subjekty["hodnota"], errors="coerce").astype(
    "Int64"
)

# přejmenování sloupců a jednoho kraje
subjekty = subjekty.rename(columns={"text1": "Kraj", "casref": "Rok", "odvetvi_txt": "Převažující činnost"})
subjekty["Kraj"] = subjekty["Kraj"].replace("Hlavní město Praha", "Praha")

# seřazení a kontrola duplicit
subjekty = subjekty.sort_values(["Kraj", "Rok"])
subjekty = subjekty.reset_index(drop=True)
subjekty

#subjekty.to_csv("ekonomicke_subjekty.csv", index=False)

C:\Users\ZŠ Tyršova\AppData\Local\Temp\ipykernel_18756\1220465501.py:12: DtypeWarning: Columns (0: aktivita_txt) have mixed types. Specify dtype option on import or set low_memory=False.
  subjekty = pd.concat([pd.read_csv(f) for f in odvetvi_roky], ignore_index=True)
